# Smoke test

Runs the whole AgriPulse pipeline end to end, from the project root.

Before running it: build the SQL Server warehouse (`build_database.bat`, or `sql/run_all.sql`; see the README) and set `AGRIPULSE_SQL_CONN` in `.env` (copy `.env.example`). Every step prints PASS / FAIL or row counts; nothing should fail.

In [ ]:
import os, sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")          # run every command from the project root
PY = sys.executable
print(Path.cwd())

In [ ]:
!{PY} -m pip -q install -r requirements.txt

Pandas mirror of Bronze -> Silver -> Gold (rebuilds `data/exports/`):

In [ ]:
!{PY} -m src.ingest.pipeline

Data-quality checks on the SQL warehouse (every check must PASS):

In [ ]:
!{PY} -m src.quality.run_checks

Semantic layer: load `semantic/metrics.yml` into SQL and validate every metric:

In [ ]:
!{PY} -m src.semantic.sync_metrics

Mining: read `gold.features_shock`, train, write predictions to `ml.*`:

In [ ]:
!{PY} -m src.mining.score

All tests:

In [ ]:
!{PY} -m pytest -q